# Dynamic Phi Model Training with GRPO

This notebook implements the same training process as the dynamic_phi.py script, allowing for interactive execution and visualization of the training process.

## Setup Logging

First, let's set up logging to track our progress.

In [1]:
import os

# Set GPU device
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
print(f"Using GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


Using GPU: 3


In [3]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
import os
import wandb
import logging
import json
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from datetime import datetime
from unsloth import is_bfloat16_supported
import torch
import sys
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Ensure the project root is in sys.path for imports
import sys
sys.path.append("/Home/stat/laschos/math/AIMO2_initial")
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker
from utils.data_preparation import prepare_combined_data
from utils.agents import (
    FULLSOLUTION_SYSTEM_PROMPT, 
    COMPLETION_SYSTEM_PROMPT,
    PROGRAMMER_SYSTEM_PROMPT
)

In [4]:
def setup_logging(model_type: str) -> logging.Logger:
    """Setup logging configuration"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"logs/{model_type}"
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger('dynamic_grpo')
    
    # Clear any existing handlers to prevent duplicate logging
    if logger.handlers:
        logger.handlers.clear()
        
    logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(
        f"{log_dir}/training_{timestamp}.log"
    )
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())
    return logger

class LoggingCallback(TrainerCallback):
    """Callback for logging training metrics"""
    def __init__(self, reward_func, logger, save_frequency=100):
        self.reward_func = reward_func
        self.save_frequency = save_frequency
        self.step = 0
        self.logger = logger
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        self.step += 1
        
        if logs and 'rewards/0' in logs and hasattr(self.reward_func, 'stats'):
            # Print detailed stats to console/log file
            self.logger.info("\n" + "="*50)
            self.logger.info(f"Step {self.step} - Reward Stats Summary:")
            
            # Get and log the stats summary
            stats_summary = self.reward_func.stats.get_summary()
            self.logger.info(stats_summary)
            self.logger.info("="*50 + "\n")
            
            # Key performance metrics for wandb
            wandb_stats = {
                'reward': logs['rewards/0'],
                'average_reward': self.reward_func.stats.reward_components.get('average_reward', 0.0),
                'total_batches': self.reward_func.stats.total_batches,
                'total_examples': self.reward_func.stats.total_examples
            }
            
            # Add dynamic reward specific metrics
            if 'solution_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['solution_reward_uses'] = self.reward_func.stats.reward_components['solution_reward_uses']
            if 'completion_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['completion_reward_uses'] = self.reward_func.stats.reward_components['completion_reward_uses']
                
            # Track example types in the batch
            if hasattr(state, 'train_dataloader') and state.train_dataloader is not None:
                try:
                    # Get current batch
                    batch_idx = (state.global_step - 1) % len(state.train_dataloader)
                    current_batch = list(state.train_dataloader)[batch_idx]
                    
                    # Count example types if available
                    if 'example_type' in current_batch:
                        example_types = current_batch['example_type']
                        solution_count = sum(1 for t in example_types if t == 'solution')
                        completion_count = sum(1 for t in example_types if t == 'completion')
                        wait_count = sum(1 for t in example_types if t == 'wait')
                        
                        wandb_stats['solution_examples'] = solution_count
                        wandb_stats['completion_examples'] = completion_count
                        wandb_stats['wait_examples'] = wait_count
                except Exception as e:
                    self.logger.warning(f"Could not track example types: {str(e)}")
            
            # Add all stats from reward_components to wandb
            for key, value in self.reward_func.stats.reward_components.items():
                wandb_stats[f'reward_components/{key}'] = value
                
            # Add group stats
            for key, value in self.reward_func.stats.group_stats.items():
                wandb_stats[f'group_stats/{key}'] = value
                
            # Add step stats
            for key, value in self.reward_func.stats.step_stats.items():
                wandb_stats[f'step_stats/{key}'] = value
                
            # Add similarity stats
            for key, value in self.reward_func.stats.similarity_stats.items():
                wandb_stats[f'similarity_stats/{key}'] = value
                
            # Add programming stats
            for key, value in self.reward_func.stats.programming_stats.items():
                wandb_stats[f'programming_stats/{key}'] = value
                
            # Add reward distribution
            if hasattr(self.reward_func.stats, 'reward_distribution') and self.reward_func.stats.reward_distribution:
                # Only log the top 10 most common rewards to avoid cluttering wandb
                sorted_rewards = sorted(
                    self.reward_func.stats.reward_distribution.items(), 
                    key=lambda x: self.reward_func.stats.reward_distribution[x[0]], 
                    reverse=True
                )[:10]
                
                for reward, count in sorted_rewards:
                    wandb_stats[f'reward_distribution/{reward}'] = count
            
            # Update logs with our metrics
            logs.update(wandb_stats)

## Main Training Setup

Now let's set up the main training configuration and components.

In [5]:
# Configuration
# Configuration
model_type = "dynamic_1"
model_name = "/Home/stat/laschos/math/AIMO2_initial/models/dynamic_0/20250306_212426"
dataset_name = "Metaskepsis/completion"

# Setup logging first
logger = setup_logging(model_type)

# Initialize config
reward_config = RewardConfig(model_type=model_type)
reward_config.group_diversity_bonus = 2  # Increased from 1.0

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"train_results/{reward_config.model_type}/{timestamp}"
wandbname = f"{model_type}, DB={reward_config.group_diversity_bonus}, {model_name}, {dataset_name}, {timestamp}"

# Initialize wandb
wandb.init(
    project="grpo",
    name=wandbname,
    config={
        "model_type": reward_config.model_type,
        "dataset": dataset_name,
        "base_reward": 3.0,
        "diversity_bonus": 0.3,
        "step_continuity_reward": 0.5
    }
)

# Initialize similarity checker first
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)
logger.info("\nInitialized DynamicReward:")
logger.info(f"Has stats object: {hasattr(reward_func, 'stats')}")

# Print initial stats configuration
if hasattr(reward_func, 'stats'):
    logger.info("Initial stats configuration:")
    for category in ['reward_components', 'group_stats', 'step_stats', 'similarity_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            logger.info(f"{category}: {stats_dict}")
else:
    logger.warning("No stats object found in reward_func!")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: artnoage (metaskepsis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Initialized DynamicReward:
Has stats object: True
Initial stats configuration:
reward_components: {'base_rewards': 0, 'step_continuity_rewards': 0, 'diversity_bonuses': 0, 'similarity_penalties': 0, 'validation_rewards': 0, 'total_length_penalty': 0.0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_rewards': 0.0, 'average_reward': 0.0, 'solution_reward_uses': 0, 'completion_reward_uses': 0, 'programming_reward_uses': 0, 'wait_examples_processed': 0, 'wait_examples_rewarded': 0, 'structure_rewards': 0, 'syntax_rewards': 0, 'execution_rewards': 0, 'correctness_rewards': 0, 'syntax_valid_solutions': 0, 'execution_valid_solutions': 0}
group_stats: {'unique_solutions': 0, 'similar_solutions': 0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_similarity': 0.0, 'diversity_bonuses': 0, 'similarity_penalties': 0}
step_stats: {'correct_step_numbering': 0, 'incorrect_step_numbering': 0, 'total_steps_completed': 0}
similarity_stats: {'unique_completions': 0, 'similar_completions': 0, 

In [ ]:
# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=4596,
    fast_inference=True,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
    gpu_memory_utilization=0.6,
    max_lora_rank=128)
    
# Function to count tokens in a string
def count_tokens(text):
    return len(tokenizer.encode(text))
    
# Calculate token counts for system prompts
solver_prompt_tokens = count_tokens(FULLSOLUTION_SYSTEM_PROMPT)
completion_prompt_tokens = count_tokens(COMPLETION_SYSTEM_PROMPT)
logger.info(f"Solver system prompt: {solver_prompt_tokens} tokens")
logger.info(f"Completion system prompt: {completion_prompt_tokens} tokens")

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)
    
def get_questions(split="train") -> Dataset:
    """Load and format dataset with full solution, completion, programming, and wait examples
    with the following distribution:
    - 35% solution examples
    - 35% programming examples
    - 15% completion examples
    - 15% wait examples
    """
    
    
    # Load the base dataset
    data = load_dataset(dataset_name, split=split)
    data= data.shuffle(seed=20)
    # Define the distribution
    distribution = {
        'solution': 0.35,
        'programming': 0.35,
        'completion': 0.15,
        'wait': 0.15
    }
    
    # Use the prepare_combined_data function with programming system prompt
    return prepare_combined_data(
        data, 
        FULLSOLUTION_SYSTEM_PROMPT, 
        COMPLETION_SYSTEM_PROMPT, 
        PROGRAMMER_SYSTEM_PROMPT,
        tokenizer, 
        distribution)

# Get the formatted dataset with all types of examples
formatted_dataset = get_questions()
# Shuffle the combined dataset
formatted_dataset = formatted_dataset.shuffle(seed=20)
# Use a reasonable number of examples
formatted_dataset = formatted_dataset.select(range(2000))

# Verify first few entries
solution_count = 0
completion_count = 0
wait_count = 0
programming_count = 0

for i in range(min(12, len(formatted_dataset))):
    entry = formatted_dataset[i]
    example_type = entry.get('example_type', 'unknown')
    
    if example_type == 'solution':
        solution_count += 1
    elif example_type == 'completion':
        completion_count += 1
    elif example_type == 'wait':
        wait_count += 1
    elif example_type == 'programming':
        programming_count += 1
        
    print(f"\nEntry {i} verification:")
    print(f"Type: {example_type}")
    print(f"Answer: {entry.get('answer')}")
    
    # Get token count for the prompt
    prompt = entry.get('prompt', '')
    prompt_tokens = count_tokens(prompt)
    print(f"Prompt tokens: {prompt_tokens}")
    
    if example_type == 'completion' and entry.get('partial_solution'):
        partial = entry.get('partial_solution')
        # Count steps in partial solution
        step_count = len(re.findall(r'<step>', partial))
        print(f"Steps in partial solution: {step_count}")
        
    elif example_type == 'wait':
        # Extract thinking section to verify wait modification
        thinking_pattern = re.compile(r'<thinking>(.*?)</thinking>', re.DOTALL)
        thinking_match = thinking_pattern.search(prompt)
    
    # Check for prompt indicators
    has_continue = 'continue' in prompt.lower()
    has_next_step = 'next step' in prompt.lower()
    has_wait = 'wait a second' in prompt.lower()
    print(f"Prompt indicators: continue={has_continue}, next_step={has_next_step}, wait={has_wait}")

print(f"\nSample ratio: {solution_count} solution examples, {completion_count} completion examples, {wait_count} wait examples, {programming_count} programming examples")

# GRPO specific training arguments
training_args = GRPOConfig(
    torch_empty_cache_steps=1,
    learning_rate=6e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=10,
    gradient_accumulation_steps=4,
    num_generations=10,
    max_prompt_length=2048,
    max_completion_length=2548,
    num_train_epochs=1,
    save_steps=50,
    max_grad_norm=0.1,
    report_to="wandb",
    output_dir=output_dir,
)

# Log the dataset structure before training
logger.info("Dataset structure before training:")
sample_example = formatted_dataset[0]
for key, value in sample_example.items():
    logger.info(f"  {key}: {type(value)} - {value}")

# Initialize trainer with reward function
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_func],
    args=training_args,
    train_dataset=formatted_dataset,
    callbacks=[LoggingCallback(reward_func=reward_func, logger=logger, save_frequency=10)]
)

# Log dataset information before training
logger.info("Dataset information before training:")
logger.info(f"Total examples: {len(formatted_dataset)}")

# Count example types in the dataset
example_types = {}
for example in formatted_dataset:
    et = example.get('example_type', 'unknown')
    example_types[et] = example_types.get(et, 0) + 1

logger.info(f"Example types in dataset: {example_types}")

# Log a sample batch structure
sample_batch = {
    'prompt': [formatted_dataset[i]['prompt'] for i in range(min(3, len(formatted_dataset)))],
    'answer': [formatted_dataset[i]['answer'] for i in range(min(3, len(formatted_dataset)))],
    'example_type': [formatted_dataset[i]['example_type'] for i in range(min(3, len(formatted_dataset)))]
}

logger.info("Sample batch structure:")
for key, value in sample_batch.items():
    if key != 'prompt':  # Skip logging the full prompts
        logger.info(f"  {key}: {value}")

# The example_type is already in the dataset, no need to add it again
# Just verify that it's present in all examples
example_type_missing = sum(1 for example in formatted_dataset if "example_type" not in example)
if example_type_missing > 0:
    logger.warning(f"Found {example_type_missing} examples without example_type field")
else:
    logger.info("All examples have example_type field correctly set")

# Print a few examples to verify example_type is set correctly
for i in range(min(5, len(formatted_dataset))):
    logger.info(f"Example {i} type: {formatted_dataset[i]['example_type']}")

INFO 03-08 08:47:52 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.8: Fast Qwen2 patching. Transformers: 4.49.0. vLLM: 0.7.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.393 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading /Home/stat/laschos/math/AIMO2_initial/models/programming_0/20250306_214045 with actual GPU utilization = 59.3%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.39 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4596. Num Sequences = 224.
Unsloth: vLLM's KV Cache can use up to 8.58 GB. Also swap space = 6 GB.
INFO 03-08 08:48:05 config.py:549] This model supports multiple tasks: {'sco

[W308 08:48:06.220572510 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 03-08 08:48:16 model_runner.py:1115] Loading model weights took 14.3620 GB
INFO 03-08 08:48:16 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 03-08 08:48:21 worker.py:267] Memory profiling takes 4.60 seconds
INFO 03-08 08:48:21 worker.py:267] the current vLLM instance can use total_gpu_memory (39.39GiB) x gpu_memory_utilization (0.59) = 23.36GiB
INFO 03-08 08:48:21 worker.py:267] model weights take 14.36GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.25GiB; the rest of the memory reserved for KV Cache is 7.65GiB.
INFO 03-08 08:48:21 executor_base.py:111] # cuda blocks: 8956, # CPU blocks: 7021
INFO 03-08 08:48:21 executor_base.py:116] Maximum concurrency for 4596 tokens per request: 31.18x
INFO 03-08 08:48:28 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error 

Capturing CUDA graph shapes: 100%|████████████████████████████████████████████| 31/31 [00:28<00:00,  1.08it/s]

INFO 03-08 08:48:56 model_runner.py:1562] Graph capturing finished in 29 secs, took 1.67 GiB
INFO 03-08 08:48:56 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 40.48 seconds



Solver system prompt: 150 tokens
Completion system prompt: 285 tokens
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x14691e680d90>>
Traceback (most recent call last):
  File "/Home/stat/laschos/.conda/envs/sloth/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
Unsloth 2025.3.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Dataset has 13238 examples with model_solutions
Found 5201 examples with valid steps (2+ steps)
Creating solution examples...
Creating programming examples...
Creating completion examples...
Found 5228 completion examples after filtering
Creating wait examples...
Found 7079 wait examples after filtering
Created 13238 full solution examples (target: 4633)
Created 13238 programming examples (target: 4633)
Created 5228 completion examples (target: 1985)
Created 7079 w

## Start Training

Now let's start the training process.

In [ ]:
 # Train
try:
    trainer.train()
    logger.info("Training completed successfully")
except Exception as e:
    logger.error(f"Training failed: {str(e)}")
    wandb.finish()
    raise

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 10 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (10 x 4 x 1) = 40
 "-____-"     Trainable parameters = 322,961,408/7,938,577,920 (4.07% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Available kwargs: ['prompts', 'id', 'data_type', 'problem', 'correct_solution', 'correct_answer', 'model_solution', 'model_answer', 'is_correct', 'attempt_number', 'total_attempts', 'answer', 'partial_solution', 'example_type']
example_type found: ['programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming', 'programming'] (type: <class 

does it True True
does it True True


Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 13.0
Used programming_reward with result: 1.7473
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 236 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Applied correctness reward: +2.500
Used programming_reward with result: 4.2476
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 151 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 7.0
Used programming_reward with result: 1.7485
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 357 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 3.0
Used programming_reward with result: 1.7464
Processing example type: programming with progr

does it True True
does it True True
does it True True
does it True True


Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 6.0
Used programming_reward with result: 1.7478
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 295 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 30.0
Used programming_reward with result: 1.7470
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 374 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 30.0
Used programming_reward with result: 1.7463
Processing example type: programming with programming_reward
Applied structure reward: +0.500
Extracted code length: 288 characters
Applied syntax reward: +0.500
Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 30.0
Used programming_reward with result: 1.7471
Processing example type: programming wi

does it True True
does it True True
does it True True
does it True True


Applied execution reward: +0.750
Incorrect answer: expected 12.0, got 36.0
Used programming_reward with result: 1.7474
Rewards before: [4.24056, 1.74733, 4.24764, 1.74849, 1.74643, 1.74777, 1.74705, 1.74626, 1.74712, 1.74737]

Reward Statistics Summary:
Training time: 0:10:44.675959
Processed 2 batches (10 examples)
Average reward: 2.246602
Reward range: [1.7463, 4.2476]

Reward Distribution:
  1.75:    8 |████████████████████████████████████████
  2.25:    0 |
  2.75:    0 |
  3.25:    0 |
  3.75:    2 |██████████

Reward Components:
  Base Rewards: 0
  Diversity Bonuses: 0
  Similarity Penalties: 0
  Base Rewards: 0
  Step Continuity Rewards: 0
  Diversity Bonuses: 0
  Similarity Penalties: 0
  Total Length Penalty: 0.033980
  Correct Answers: 0
  Incorrect Answers: 0
  Total Rewards: 44.932040
  Average Reward: 2.246602
  Structure Rewards: 10
  Syntax Rewards: 10
  Execution Rewards: 10
  Correctness Rewards: 2
  Total Length Penalty: 0.033980
  Correct Solutions: 2
  Syntax Valid 

Unsloth: Will smartly offload gradients to save VRAM!


## Save Model

Finally, let's save the trained model.

In [ ]:
# Save model
try:
    models_dir = "models"
    os.makedirs(os.path.join(models_dir, reward_config.model_type), exist_ok=True)
    model_output_dir = os.path.join(models_dir, reward_config.model_type, timestamp)
    model.save_pretrained_merged(model_output_dir, tokenizer, save_method="merged_16bit")
    logger.info(f"Merged model saved to {model_output_dir}")
    print(f"Model saved to {model_output_dir}")
except Exception as e:
    logger.error(f"Failed to save model: {str(e)}")
    print(f"Error saving model: {str(e)}")
finally:
    if use_wandb:
        wandb.finish()
        print("Wandb logging finished")

## Conclusion

This notebook has demonstrated the complete training process for a Qwen model using GRPO with dynamic rewards. We've seen how to:

1. Configure the reward function
2. Prepare a dataset with different example types
3. Initialize and configure the model with LoRA
4. Set up the GRPO training process
5. Train the model and visualize the results
6. Save the trained model

This interactive approach allows for better monitoring and understanding of the training process compared to running the script directly.